# Partie 3 : Spark SQL et DataFrames

On refait l'analyse de la Partie 2 avec l'API structurée, puis on l'enrichit : jointure avec le référentiel
des commerçants, UDF, écriture Parquet partitionnée et lecture du plan d'exécution (Catalyst).

In [16]:
import json, os, time
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from sahlpay_spark import get_spark, chrono, mediane, list_lake, LAKE

spark = get_spark("SahlPay-Partie3-SQL")
HISTORY = f"{LAKE}/bronze/history/transactions_history.jsonl"
MERCHANTS = f"{LAKE}/reference/merchants.json"
SILVER = f"{LAKE}/silver/transactions_history"
OUT = os.environ.get("SAHLPAY_OUT", "../outputs")
os.makedirs(OUT, exist_ok=True)
temps = {}

## 1. Lecture avec un schéma explicite (`StructType`, sans inférence)

Sans schéma, Spark lit tout le fichier une première fois pour deviner les types. Avec un schéma explicite :
une seule lecture, des types garantis (`amount_mad` en double, `event_time` en timestamp).

In [2]:
tx_schema = StructType([
    StructField("tx_id",        StringType(),    True),
    StructField("type",         StringType(),    True),
    StructField("payer_wallet", StringType(),    True),
    StructField("payee_id",     StringType(),    True),
    StructField("device_id",    StringType(),    True),
    StructField("city",         StringType(),    True),
    StructField("channel",      StringType(),    True),
    StructField("status",       StringType(),    True),
    StructField("amount_mad",   DoubleType(),    True),
    StructField("event_time",   TimestampType(), True),
])

merchant_schema = StructType([
    StructField("merchant_id", StringType(), True),
    StructField("name",        StringType(), True),
    StructField("city",        StringType(), True),
    StructField("category",    StringType(), True),
    StructField("latitude",    DoubleType(), True),
    StructField("longitude",   DoubleType(), True),
])

tx = (spark.read
      .schema(tx_schema)
      .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss[.SSS]XXX")
      .option("mode", "PERMISSIVE")
      .json(HISTORY))

merchants = (spark.read
             .schema(merchant_schema)
             # .option("multiLine", True)   # à activer si merchants.json est un tableau JSON [ {...}, ... ]
             .json(MERCHANTS))

tx.printSchema()
print("Transactions :", tx.count(), "| commerçants :", merchants.count())
tx.show(3, truncate=False)

26/11/25 21:47:29 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- tx_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- payer_wallet: string (nullable = true)
 |-- payee_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



Transactions : 50000 | commerçants : 300
+-----------+----+------------+--------+---------+----------+-------+----------+----------+-------------------+
|tx_id      |type|payer_wallet|payee_id|device_id|city      |channel|status    |amount_mad|event_time         |
+-----------+----+------------+--------+---------+----------+-------+----------+----------+-------------------+
|TX-00000001|P2M |W-14188     |M-245   |D-4dae40 |Casablanca|APP    |INITIATED |157.06    |2026-11-01 12:57:12|
|TX-00000001|P2M |W-14188     |M-245   |D-4dae40 |Casablanca|APP    |AUTHORIZED|157.06    |2026-11-01 12:57:13|
|TX-00000001|P2M |W-14188     |M-245   |D-4dae40 |Casablanca|APP    |COMPLETED |157.06    |2026-11-01 12:57:14|
+-----------+----+------------+--------+---------+----------+-------+----------+----------+-------------------+
only showing top 3 rows



## 2. Montant par ville : API DataFrame, puis SQL

In [3]:
completed = tx.filter(F.col("status") == "COMPLETED")

par_ville_df = (completed.groupBy("city")
                .agg(F.round(F.sum("amount_mad"), 2).alias("montant_total_mad"),
                     F.count("*").alias("nb_transactions"),
                     F.round(F.avg("amount_mad"), 2).alias("montant_moyen_mad"))
                .orderBy(F.desc("montant_total_mad")))

with chrono("DataFrame - total + moyenne par ville", temps):
    res_df = par_ville_df.toPandas()
res_df

[temps] DataFrame - total + moyenne par ville : 1.640 s


,city,montant_total_mad,nb_transactions,montant_moyen_mad
0,Casablanca,1471886.03,5010,293.79
1,Rabat,697455.61,2695,258.80
2,Marrakech,470642.44,1741,270.33
3,Fes,411967.55,1481,278.17
4,Tanger,403067.27,1418,284.25
5,Agadir,287146.05,1089,263.68
6,Meknes,234613.87,554,423.49
7,Kenitra,231810.18,602,385.07
8,Oujda,211568.72,532,397.69
9,Tetouan,204449.53,676,302.44


In [4]:
tx.createOrReplaceTempView("transactions")

requete = '''
SELECT city,
       ROUND(SUM(amount_mad), 2) AS montant_total_mad,
       COUNT(*)                  AS nb_transactions,
       ROUND(AVG(amount_mad), 2) AS montant_moyen_mad
FROM transactions
WHERE status = 'COMPLETED'
GROUP BY city
ORDER BY montant_total_mad DESC
'''
with chrono("SQL - total + moyenne par ville", temps):
    res_sql = spark.sql(requete).toPandas()
res_sql

[temps] SQL - total + moyenne par ville : 0.729 s


,city,montant_total_mad,nb_transactions,montant_moyen_mad
0,Casablanca,1471886.03,5010,293.79
1,Rabat,697455.61,2695,258.80
2,Marrakech,470642.44,1741,270.33
3,Fes,411967.55,1481,278.17
4,Tanger,403067.27,1418,284.25
5,Agadir,287146.05,1089,263.68
6,Meknes,234613.87,554,423.49
7,Kenitra,231810.18,602,385.07
8,Oujda,211568.72,532,397.69
9,Tetouan,204449.53,676,302.44


In [5]:
# Contrôle : les trois méthodes (RDD, DataFrame, SQL) donnent les mêmes chiffres
chemin_p2 = f"{OUT}/partie2_villes.csv"
if os.path.exists(chemin_p2):
    rdd_res = pd.read_csv(chemin_p2).set_index("ville")["montant_total_mad"]
    df_res = res_df.set_index("city")["montant_total_mad"]
    ecart = (rdd_res - df_res).abs().max()
    print("Écart maximal RDD / DataFrame :", round(float(ecart), 4), "MAD")
print("DataFrame et SQL identiques :", res_df.equals(res_sql))

Écart maximal RDD / DataFrame : 0.0 MAD
DataFrame et SQL identiques : True


## 3. Jointure des paiements P2M avec le référentiel : montant par catégorie et par ville

In [6]:
p2m = completed.filter(F.col("type") == "P2M")
ref = merchants.select("merchant_id", "category", F.col("city").alias("merchant_city"), "name")

jointure = p2m.join(ref, p2m.payee_id == ref.merchant_id, "inner")

par_categorie_ville = (jointure.groupBy("category", "city")
                       .agg(F.count("*").alias("nb_paiements"),
                            F.round(F.sum("amount_mad"), 2).alias("montant_total_mad"))
                       .orderBy("city", F.desc("montant_total_mad")))

with chrono("DataFrame - jointure + agrégation catégorie x ville", temps):
    res_cat = par_categorie_ville.toPandas()
res_cat.to_csv(f"{OUT}/partie3_categorie_ville.csv", index=False)
print(len(res_cat), "lignes (catégorie x ville)")
res_cat.head(15)

[temps] DataFrame - jointure + agrégation catégorie x ville : 1.015 s
72 lignes (catégorie x ville)


,category,city,nb_paiements,montant_total_mad
0,e-commerce,Agadir,127,23574.14
1,telecom,Agadir,124,18867.02
2,carburant,Agadir,137,18164.88
3,pharmacie,Agadir,101,14071.18
4,epicerie,Agadir,60,9413.31
5,restauration,Agadir,52,7775.28
6,habillement,Agadir,32,5907.28
7,transport,Agadir,34,5155.44
8,carburant,Casablanca,542,85330.61
9,habillement,Casablanca,513,83849.93


In [7]:
# Vue plus lisible pour le rapport : catégories en colonnes, villes en lignes (montants en MAD)
res_cat.pivot_table(index="city", columns="category", values="montant_total_mad", aggfunc="sum").round(0)

category,carburant,e-commerce,epicerie,habillement,pharmacie,restauration,telecom,transport
city,,,,,,,,
Agadir,18165.0,23574.0,9413.0,5907.0,14071.0,7775.0,18867.0,5155.0
Casablanca,85331.0,41112.0,45373.0,83850.0,53902.0,77598.0,42798.0,76327.0
Fes,8046.0,27489.0,17051.0,20829.0,20106.0,25507.0,12419.0,13723.0
Kenitra,12536.0,10974.0,18378.0,8584.0,NaN,NaN,8272.0,5028.0
Marrakech,15151.0,23860.0,17946.0,28795.0,42584.0,18002.0,17101.0,13059.0
Meknes,4459.0,8831.0,4404.0,3189.0,6856.0,4027.0,NaN,14932.0
Oujda,4571.0,4704.0,8948.0,16265.0,NaN,7988.0,NaN,NaN
Rabat,18899.0,50874.0,28157.0,42070.0,62214.0,22778.0,14928.0,56916.0
Tanger,25954.0,NaN,21574.0,17291.0,34686.0,3688.0,17839.0,24538.0


## 4. UDF `amount_band` contre `when` / `otherwise`

- `micro` : moins de 50 MAD
- `standard` : de 50 à 2 000 MAD
- `eleve` : plus de 2 000 MAD

In [8]:
@F.udf(returnType=StringType())
def amount_band(amount_mad):
    if amount_mad is None:
        return None
    if amount_mad < 50:
        return "micro"
    if amount_mad <= 2000:
        return "standard"
    return "eleve"

band_natif = (F.when(F.col("amount_mad") < 50, "micro")
               .when(F.col("amount_mad") <= 2000, "standard")
               .when(F.col("amount_mad").isNotNull(), "eleve"))

avec_udf = tx.withColumn("amount_band", amount_band(F.col("amount_mad")))
sans_udf = tx.withColumn("amount_band", band_natif)

def mesurer(df, repetitions=5):
    durees = []
    for _ in range(repetitions):
        debut = time.perf_counter()
        df.groupBy("amount_band").count().collect()
        durees.append(time.perf_counter() - debut)
    return round(sorted(durees)[len(durees) // 2], 3)       # médiane

mesurer(sans_udf, 1)                                        # échauffement (JVM, cache du fichier)
temps["UDF Python - amount_band (médiane de 5)"] = mesurer(avec_udf)
temps["when/otherwise - amount_band (médiane de 5)"] = mesurer(sans_udf)
print({k: v for k, v in temps.items() if "amount_band" in k})

sans_udf.groupBy("amount_band").count().orderBy("amount_band").show()

{'UDF Python - amount_band (médiane de 5)': 0.355, 'when/otherwise - amount_band (médiane de 5)': 0.251}
+-----------+-----+
|amount_band|count|
+-----------+-----+
|      eleve|  856|
|      micro| 9515|
|   standard|39629|
+-----------+-----+



In [9]:
# Les deux versions donnent le même résultat
diff = (avec_udf.select("tx_id", "status", "amount_band")
        .exceptAll(sans_udf.select("tx_id", "status", "amount_band")).count())
print("Lignes différentes entre UDF et when/otherwise :", diff)

# Le plan montre pourquoi l'UDF est plus lente : BatchEvalPython = envoi des lignes à un processus Python
print("---- Plan avec UDF ----")
avec_udf.select("amount_band").explain()
print("---- Plan sans UDF ----")
sans_udf.select("amount_band").explain()

Lignes différentes entre UDF et when/otherwise : 0
---- Plan avec UDF ----
== Physical Plan ==
*(1) Project [pythonUDF0#559 AS amount_band#259]
+- BatchEvalPython [amount_band(amount_mad#8)#258], [pythonUDF0#559]
   +- FileScan json [amount_mad#8] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/transactions_history.jsonl], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<amount_mad:double>


---- Plan sans UDF ----
== Physical Plan ==
*(1) Project [CASE WHEN (amount_mad#8 < 50.0) THEN micro WHEN (amount_mad#8 <= 2000.0) THEN standard WHEN isnotnull(amount_mad#8) THEN eleve END AS amount_band#271]
+- FileScan json [amount_mad#8] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/transactions_history.jsonl], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<amount_mad:double>




## 5. Écriture du résultat nettoyé en Parquet (couche Silver), partitionné par `city`

Nettoyage : suppression des doublons (`tx_id`, `status`), des lignes sans identifiant, sans ville ou avec
un montant invalide ; ajout de `amount_band` et de la date.

In [10]:
silver = (tx.dropDuplicates(["tx_id", "status"])
            .filter(F.col("tx_id").isNotNull() & F.col("city").isNotNull()
                    & F.col("amount_mad").isNotNull() & (F.col("amount_mad") > 0))
            .withColumn("amount_band", band_natif)
            .withColumn("date", F.to_date("event_time")))

with chrono("Écriture Parquet Silver partitionné par city", temps):
    silver.write.mode("overwrite").partitionBy("city").parquet(SILVER)

relu = spark.read.parquet(SILVER)
print("Lignes Bronze :", tx.count(), "| lignes Silver :", relu.count())
relu.groupBy("city").count().orderBy(F.desc("count")).show()
list_lake(spark, SILVER, depth=1)

[temps] Écriture Parquet Silver partitionné par city : 5.114 s
Lignes Bronze : 50000 | lignes Silver : 48972
+----------+-----+
|      city|count|
+----------+-----+
|Casablanca|15524|
|     Rabat| 8349|
| Marrakech| 5406|
|       Fes| 4520|
|    Tanger| 4418|
|    Agadir| 3396|
|   Tetouan| 2107|
|   Kenitra| 1878|
|    Meknes| 1753|
|     Oujda| 1621|
+----------+-----+

s3a://lake/silver/transactions_history
  _SUCCESS  (0 octets)
  city=Agadir/
  city=Casablanca/
  city=Fes/
  city=Kenitra/
  city=Marrakech/
  city=Meknes/
  city=Oujda/
  city=Rabat/
  city=Tanger/
  city=Tetouan/


## 6. Plan d'exécution de la jointure : `explain(True)`

À identifier dans le rapport :
- **Parsed / Analyzed Logical Plan** : la requête telle qu'écrite, puis résolue (colonnes, types) ;
- **Optimized Logical Plan** : après Catalyst (filtres poussés vers la lecture, colonnes inutiles retirées,
  `isnotnull` ajouté sur la clé de jointure) ;
- **Physical Plan** : la stratégie de jointure choisie.

In [11]:
jointure.explain(True)

== Parsed Logical Plan ==
Join Inner, (payee_id#3 = merchant_id#20)
:- Filter (type#1 = P2M)
:  +- Filter (status#7 = COMPLETED)
:     +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] json
+- Project [merchant_id#20, category#23, city#22 AS merchant_city#186, name#21]
   +- Relation [merchant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
tx_id: string, type: string, payer_wallet: string, payee_id: string, device_id: string, city: string, channel: string, status: string, amount_mad: double, event_time: timestamp, merchant_id: string, category: string, merchant_city: string, name: string
Join Inner, (payee_id#3 = merchant_id#20)
:- Filter (type#1 = P2M)
:  +- Filter (status#7 = COMPLETED)
:     +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] json
+- Project [merchant_id#20, category#23, city#22 AS merc

In [12]:
# Sauvegarde du plan pour le rapport
plan = jointure._jdf.queryExecution().toString()
with open(f"{OUT}/partie3_explain_jointure.txt", "w") as f:
    f.write(plan)
strategie = "BroadcastHashJoin" if "BroadcastHashJoin" in plan else ("SortMergeJoin" if "SortMergeJoin" in plan else "autre")
print("Stratégie de jointure physique :", strategie)
print("Seuil de broadcast automatique :", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

Stratégie de jointure physique : BroadcastHashJoin
Seuil de broadcast automatique : 10485760b


In [13]:
# Pour comparaison : sans broadcast, Spark choisit un SortMergeJoin (shuffle des deux côtés + tri).
# Le DataFrame doit être reconstruit après le changement de configuration (le plan est figé à sa création).
def construire_jointure():
    p2m_ = tx.filter((F.col("status") == "COMPLETED") & (F.col("type") == "P2M"))
    ref_ = merchants.select("merchant_id", "category")
    return (p2m_.join(ref_, p2m_.payee_id == ref_.merchant_id, "inner")
                .groupBy("category", "city").agg(F.sum("amount_mad")))

spark.conf.set("spark.sql.adaptive.enabled", "false")          # pour que le plan affiché soit le plan exécuté
for seuil, nom in [("-1", "SortMergeJoin (broadcast désactivé)"), (str(10 * 1024 * 1024), "BroadcastHashJoin (seuil 10 Mo)")]:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", seuil)
    df_j = construire_jointure()
    plan_physique = df_j._jdf.queryExecution().executedPlan().toString()
    trouve = "SortMergeJoin" if "SortMergeJoin" in plan_physique else "BroadcastHashJoin"
    mediane(lambda: df_j.collect(), 3, f"Jointure {nom}", temps)
    print("   stratégie dans le plan :", trouve)
spark.conf.set("spark.sql.adaptive.enabled", "true")

[temps] Jointure SortMergeJoin (broadcast désactivé) : mediane 0.123 s sur 3 executions
   stratégie dans le plan : SortMergeJoin
[temps] Jointure BroadcastHashJoin (seuil 10 Mo) : mediane 0.096 s sur 3 executions
   stratégie dans le plan : BroadcastHashJoin


## 7. Comparaison des temps : RDD (Partie 2), DataFrame et SQL (Partie 3)

Même traitement dans les trois cas (lecture du fichier, filtre `COMPLETED`, total et moyenne par ville),
sans cache, médiane de 3 exécutions.

In [14]:
from sahlpay_spark import mediane

def pipeline_df():
    (spark.read.schema(tx_schema).option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss[.SSS]XXX").json(HISTORY)
          .filter(F.col("status") == "COMPLETED")
          .groupBy("city")
          .agg(F.sum("amount_mad"), F.count("*"), F.avg("amount_mad"))
          .collect())

def pipeline_sql():
    spark.sql(requete).collect()

mediane(pipeline_df, 3, "DataFrame - pipeline complet depuis le fichier", temps)
mediane(pipeline_sql, 3, "SQL - pipeline complet depuis le fichier", temps)

chemin_t2 = f"{OUT}/temps_partie2.json"
temps_p2 = json.load(open(chemin_t2)) if os.path.exists(chemin_t2) else {}
comparaison = pd.DataFrame(
    [("Total + moyenne par ville (sans cache, médiane de 3)",
      temps_p2.get("RDD - pipeline complet depuis le fichier (sans cache)"),
      temps.get("DataFrame - pipeline complet depuis le fichier"),
      temps.get("SQL - pipeline complet depuis le fichier"))],
    columns=["traitement", "RDD (s)", "DataFrame (s)", "SQL (s)"])
with open(f"{OUT}/temps_partie3.json", "w") as f:
    json.dump(temps, f, indent=2)
display(comparaison)
pd.Series(temps, name="secondes")

[temps] DataFrame - pipeline complet depuis le fichier : mediane 0.305 s sur 3 executions
[temps] SQL - pipeline complet depuis le fichier : mediane 0.330 s sur 3 executions


,traitement,RDD (s),DataFrame (s),SQL (s)
0,"Total + moyenne par ville (sans cache, médiane...",0.581,0.305,0.33


DataFrame - total + moyenne par ville                  1.640
SQL - total + moyenne par ville                        0.729
DataFrame - jointure + agrégation catégorie x ville    1.015
UDF Python - amount_band (médiane de 5)                0.355
when/otherwise - amount_band (médiane de 5)            0.251
Écriture Parquet Silver partitionné par city           5.114
Jointure SortMergeJoin (broadcast désactivé)           0.123
Jointure BroadcastHashJoin (seuil 10 Mo)               0.096
DataFrame - pipeline complet depuis le fichier         0.305
SQL - pipeline complet depuis le fichier               0.330
Name: secondes, dtype: float64

**Interprétation :** sur 50 000 lignes, les écarts restent faibles car le coût fixe (planification, lancement des
tâches) domine. L'avantage des DataFrames grandit avec le volume : Catalyst optimise le plan, Tungsten stocke les
données en binaire compact hors des objets Python, et aucune ligne ne passe par un processus Python, alors que
chaque `lambda` d'un RDD PySpark sérialise les lignes entre la JVM et Python.

Captures à faire : sortie de `explain(True)` ; Spark UI → onglet **SQL / DataFrame** → requête de la jointure
(on y voit `BroadcastExchange` et `BroadcastHashJoin`) ; console MinIO → `lake/silver/transactions_history/city=…/`.

In [15]:
# spark.stop()   # après les captures de la Spark UI